# 05 · Fine-tuning BERT-base (transfer learning)

**Input:** `train/val/test.pkl`, `class_weights.json`. **Output:** `outputs/bert/` and `outputs/predictions/bert_fine_tune.*`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from src.config import CFG, DEVICE, BERT_DIR, LABELS, ON_GPU, SEED, set_seed, setup_notebook
from src.data import load_class_weights, load_splits, make_collate_transformer, make_loader
from src.evaluation import plot_confusion, predict_transformer, report, save_predictions
from src.models import count_params

setup_notebook()

RETRAIN = False

HIST_PATH, STEPS_PATH, INFO_PATH = BERT_DIR / 'history.csv', BERT_DIR / 'step_losses.npy', BERT_DIR / 'train_info.json'
TRAIN_MODEL = RETRAIN or not (BERT_DIR / 'config.json').exists()
print('Fine-tune BERT' if TRAIN_MODEL else f'Model found -> loading from {BERT_DIR}')

train_df, val_df, test_df = load_splits()
y_test = test_df['label'].values
CLASS_WEIGHTS_T = torch.tensor(load_class_weights(), device=DEVICE)
MAX_LEN = CFG['trf_max_len']

## 1. Idea

## 2. Data and WordPiece tokenisation

In [ ]:
def stratified_subset(frame, n):
    if n is None or n >= len(frame):
        return frame.reset_index(drop=True)
    return train_test_split(frame, train_size=n, stratify=frame['label'], random_state=SEED)[0].reset_index(drop=True)

trf_train = stratified_subset(train_df, CFG['trf_train_size'])
trf_val = stratified_subset(val_df, CFG['trf_val_size'])
tokenizer = AutoTokenizer.from_pretrained(CFG['trf_name'] if TRAIN_MODEL else BERT_DIR)

print(f'Fine-tuning on {len(trf_train):,} train reviews, selecting the epoch on {len(trf_val):,} validation reviews, max_len = {MAX_LEN} tokens')
print('WordPiece example:', tokenizer.tokenize("The unbelievably crunchy granola wasn't overpriced!"))
sub_lens = np.array([len(x) for x in tokenizer(list(trf_train['text'].sample(min(3000, len(trf_train)), random_state=SEED)))['input_ids']])
print(f'Subword length: median {np.median(sub_lens):.0f}, p90 {np.percentile(sub_lens, 90):.0f} '
      f'| share truncated at {MAX_LEN}: {(sub_lens > MAX_LEN).mean():.1%}')

In [ ]:
t0 = time.time()
train_ids = tokenizer(list(trf_train['text']), truncation=True, max_length=MAX_LEN)['input_ids']
print(f'Tokenised in {time.time() - t0:.0f}s')

collate_trf = make_collate_transformer(tokenizer.pad_token_id)
train_loader = make_loader(train_ids, trf_train['label'].values, CFG['trf_batch_size'], shuffle=True, collate_fn=collate_trf)
_, ids_b, mask_b, _ = next(iter(train_loader))
print(f'Batches per epoch: {len(train_loader):,} | one batch: input_ids={tuple(ids_b.shape)}, attention_mask={tuple(mask_b.shape)}')

## 3. Model and optimiser

In [ ]:
set_seed(SEED)
if TRAIN_MODEL:
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG['trf_name'], num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)}).to(DEVICE)
else:
    model = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to(DEVICE)
n_params = count_params(model)
print(f'Parameters: {n_params:,}')

no_decay = ['bias', 'LayerNorm.weight']
param_groups = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': CFG['trf_weight_decay']},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = torch.optim.AdamW(param_groups, lr=CFG['trf_lr'])
total_steps = len(train_loader) * CFG['trf_epochs']
scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG['trf_warmup'] * total_steps), total_steps)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_T)
scaler = torch.amp.GradScaler('cuda', enabled=ON_GPU)
print(f'{len(train_loader):,} steps/epoch x {CFG["trf_epochs"]} epochs = {total_steps:,} steps')

## 4. Fine-tuning

In [ ]:
if TRAIN_MODEL:
    history, step_losses = [], []
    best_f1 = -1.0
    t_start = time.time()
    for epoch in range(1, CFG['trf_epochs'] + 1):
        t0 = time.time()
        model.train()
        for _, input_ids, attention_mask, y in tqdm(train_loader, desc=f'epoch {epoch}', leave=False):
            input_ids, attention_mask, y = input_ids.to(DEVICE), attention_mask.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=ON_GPU):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step_losses.append(loss.item())

        val_pred = predict_transformer(model, tokenizer, trf_val['text'], MAX_LEN, desc='validation').argmax(1)
        val_acc, val_f1 = accuracy_score(trf_val['label'], val_pred), f1_score(trf_val['label'], val_pred, average='macro')
        epoch_loss = np.mean(step_losses[-len(train_loader):])
        history.append({'epoch': epoch, 'train_loss': epoch_loss, 'val_acc': val_acc, 'val_macro_f1': val_f1})
        print(f'Epoch {epoch}: train_loss={epoch_loss:.4f} | val_acc={val_acc:.4f} | val_macro_F1={val_f1:.4f} | {(time.time() - t0) / 60:.1f} min')
        if val_f1 > best_f1:
            best_f1 = val_f1
            model.save_pretrained(BERT_DIR)
            tokenizer.save_pretrained(BERT_DIR)
        pd.DataFrame(history).to_csv(HIST_PATH, index=False)
        torch.cuda.empty_cache()
    train_time = time.time() - t_start
    model = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to(DEVICE)
    hist = pd.DataFrame(history)
    hist.to_csv(HIST_PATH, index=False)
    np.save(STEPS_PATH, np.array(step_losses))
    INFO_PATH.write_text(json.dumps({'train_time_s': train_time, 'best_val_macro_f1': best_f1,
                                     'n_train': len(trf_train), 'max_len': MAX_LEN}, indent=2))
else:
    hist = pd.read_csv(HIST_PATH) if HIST_PATH.exists() else None
    step_losses = np.load(STEPS_PATH).tolist() if STEPS_PATH.exists() else []
    info = json.loads(INFO_PATH.read_text()) if INFO_PATH.exists() else {}
    train_time, best_f1 = info.get('train_time_s'), info.get('best_val_macro_f1', float('nan'))
    print(f'Loaded {BERT_DIR}')
print(f'Fine-tuning time: {(train_time or 0) / 60:.1f} min | best validation Macro-F1: {best_f1:.4f}')

In [ ]:
if hist is not None and len(step_losses):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    window = max(1, len(step_losses) // 50)
    axes[0].plot(step_losses, alpha=.25, color='#4c72b0', label='step loss')
    axes[0].plot(pd.Series(step_losses).rolling(window, min_periods=1).mean(), color='#4c72b0', label=f'moving average ({window} steps)')
    steps_per_epoch = len(step_losses) // len(hist)
    for e in range(1, len(hist)):
        axes[0].axvline(e * steps_per_epoch, ls='--', color='gray', lw=1)
    axes[0].set(title='BERT – train loss per step', xlabel='Step (batch)', ylabel='Loss')
    axes[0].legend()
    axes[1].plot(hist['epoch'], hist['val_acc'], 'o-', label='val accuracy')
    axes[1].plot(hist['epoch'], hist['val_macro_f1'], 's-', label='val Macro-F1')
    axes[1].set(title='BERT – validation metrics', xlabel='Epoch')
    axes[1].xaxis.get_major_locator().set_params(integer=True)
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    display(hist.round(4))

## 5. Test-set evaluation

In [ ]:
t0 = time.time()
test_probs = predict_transformer(model, tokenizer, test_df['text'], MAX_LEN,
                                 batch_size=128 if ON_GPU else 64, desc='BERT test')
infer_time = time.time() - t0
test_pred = test_probs.argmax(1)
report('BERT fine-tune', y_test, test_pred)
plot_confusion(y_test, test_pred, 'BERT fine-tune – confusion matrix (test)')
plt.show()

save_predictions('BERT fine-tune', test_probs,
                 {'train_time_s': train_time, 'infer_time_s': infer_time, 'n_params': n_params})